## Shoaib Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [ ]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import _canon, load_contracts, to_continuous_stream, run_qa_checks, check_sample_integrity

C       = load_contracts(ROOT)
SCHEMA  = C["SCHEMA"]
RAW2ID  = C["RAW2ID"]
ID2NAME = C["ID2NAME"]
UNKNOWN_ID = C["UNKNOWN_ID"]
TARGET_HZ  = C["TARGET_HZ"]
CLEANED    = C["CLEANED"]

RAW_SHO  = ROOT / "data" / "raw_data" / "Shoaib"
CSV_PATH = RAW_SHO / "smartphoneatwrist.csv"

print("CSV_PATH:", CSV_PATH)
print("TARGET_HZ:", TARGET_HZ)
print("UNKNOWN_ID:", UNKNOWN_ID)

CSV_PATH: /home/aidan/IMU_LM_Data/data/raw_data/Shoaib/smartphoneatwrist.csv
TARGET_HZ: 50
UNKNOWN_ID: 9000


### Step 1. Ingest, preporccess and map the data 

In [6]:
# ============================
# STEP 1 — Load + rebuild timestamps @50Hz per contiguous label segment
#   Output columns:
#     dataset, subject_id, session_id, timestamp_ns,
#     acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z,
#     dataset_activity_id, dataset_activity_label
# ============================

SHOAIB_CODE2LABEL = {
    11111: "walk",
    11112: "stand",
    11113: "jog",
    11114: "sit",
    11115: "bike",
    11116: "upstairs",
    11117: "downstairs",
    11118: "type",
    11119: "write",
    11120: "coffee",
    11121: "talk",
    11122: "smoke",
    11123: "eat",
}

def load_shoaib_50hz_native(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, header=None)
    if df.shape[1] < 14:
        raise ValueError(f"Expected >=14 columns, got {df.shape[1]}")

    out = pd.DataFrame({
        "acc_x":  df[1].astype("float32"),
        "acc_y":  df[2].astype("float32"),
        "acc_z":  df[3].astype("float32"),
        "gyro_x": df[7].astype("float32"),
        "gyro_y": df[8].astype("float32"),
        "gyro_z": df[9].astype("float32"),
        "dataset_activity_id": df[13].astype("int32"),
    })

    out["dataset_activity_label"] = (
        out["dataset_activity_id"]
        .map(SHOAIB_CODE2LABEL)
        .fillna("unknown")
        .astype("string")
    )

    # contiguous-run segmentation (helper; NOT returned)
    seg = out["dataset_activity_id"].ne(out["dataset_activity_id"].shift(1)).cumsum().astype("int64")

    # identifiers
    out.insert(0, "dataset", "shoaib")
    out.insert(1, "subject_id", "subj_000")
    out.insert(2, "session_id", seg.map(lambda k: f"seg_{int(k):06d}").astype("string"))

    # timestamps: restart at 0 inside each segment, fixed 50Hz cadence
    step_ns = int(round(1e9 / TARGET_HZ))  # 20,000,000 at 50Hz
    i_in_seg = out.groupby("session_id").cumcount().astype("int64")
    out.insert(3, "timestamp_ns", (i_in_seg * step_ns).astype("int64"))

    # finalize dtypes
    out["dataset_activity_id"] = out["dataset_activity_id"].clip(-32768, 32767).astype("int16")

    # IMPORTANT: return only the native frame columns (WISDM-style)
    return out[[
        "dataset", "subject_id", "session_id", "timestamp_ns",
        "acc_x", "acc_y", "acc_z",
        "gyro_x", "gyro_y", "gyro_z",
        "dataset_activity_id", "dataset_activity_label",
    ]]

shoaib_50_native = load_shoaib_50hz_native(CSV_PATH)
print("Rows:", len(shoaib_50_native))
print("Sessions:", shoaib_50_native["session_id"].nunique())
shoaib_50_native.head(3)


Rows: 1170000
Sessions: 13


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,dataset_activity_id,dataset_activity_label
0,shoaib,subj_000,seg_000001,0,4.4266,-7.0281,1.3076,-0.136220,0.15394,-0.132560,11111,walk
1,shoaib,subj_000,seg_000001,20000000,4.7126,-8.5127,1.5936,-0.059559,0.26267,-0.076053,11111,walk
2,shoaib,subj_000,seg_000001,40000000,5.0531,-9.4389,1.4846,-0.054978,0.29352,0.097128,11111,walk


### Step 2. Map the data and audit the mapping

In [7]:
# ============================
# STEP 2 — Mapping audit (native → global) using activity_mapping.json
# ============================
if shoaib_50_native.empty:
    raise SystemExit("No Shoaib rows after STEP 1. Check CSV path / parsing.")

raw_counts = (
    shoaib_50_native["dataset_activity_label"]
    .astype("string")
    .map(_canon)
    .value_counts()
    .rename_axis("raw_label")
    .reset_index(name="count")
)

raw_counts["mapped_id"] = raw_counts["raw_label"].map(RAW2ID).fillna(UNKNOWN_ID).astype(int)
raw_counts["mapped_nm"] = raw_counts["mapped_id"].map(lambda x: ID2NAME.get(int(x), "other"))

unmapped = raw_counts.loc[raw_counts["mapped_id"] == UNKNOWN_ID]
print(f"[SHOAIB] Unique raw labels: {len(raw_counts)} | Unmapped: {len(unmapped)}")

total_ct = int(raw_counts["count"].sum())
mapped_ct = int(raw_counts.loc[raw_counts["mapped_id"] != UNKNOWN_ID, "count"].sum())
print(f"coverage={100.0*mapped_ct/max(total_ct,1):.2f}%  (mapped={mapped_ct:,}/{total_ct:,})")

if not unmapped.empty:
    print("\nUnmapped raw labels (should be empty if canon labels exist in mapping.json):")
    print(unmapped.sort_values("count", ascending=False).to_string(index=False))

raw_counts


[SHOAIB] Unique raw labels: 13 | Unmapped: 1
coverage=92.31%  (mapped=1,080,000/1,170,000)

Unmapped raw labels (should be empty if canon labels exist in mapping.json):
raw_label  count  mapped_id mapped_nm
    smoke  90000       9000     other


,raw_label,count,mapped_id,mapped_nm
0,walk,90000,2,walk
1,stand,90000,1,posture_stationary
2,jog,90000,3,run_jog
3,sit,90000,1,posture_stationary
4,bike,90000,5,cycle
5,upstairs,90000,4,stairs
6,downstairs,90000,4,stairs
7,type,90000,14,adl_desk_device
8,write,90000,14,adl_desk_device
9,coffee,90000,15,adl_food


### Step 3. Build and clean dataset in stream json fromat

In [ ]:
# ============================
# STEP 3 — Add global_* and schema-order (using shared to_continuous_stream)
# ============================
shoaib_df = to_continuous_stream(shoaib_50_native, SCHEMA, RAW2ID, ID2NAME, UNKNOWN_ID)
print("SHOAIB unified rows:", len(shoaib_df))
shoaib_df.head(3)

SHOAIB unified rows: 1170000


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,global_activity_id,global_activity_label,dataset_activity_id,dataset_activity_label
0,shoaib,subj_000,seg_000001,0,4.4266,-7.0281,1.3076,-0.136220,0.15394,-0.132560,2,walk,11111,walk
1,shoaib,subj_000,seg_000001,20000000,4.7126,-8.5127,1.5936,-0.059559,0.26267,-0.076053,2,walk,11111,walk
2,shoaib,subj_000,seg_000001,40000000,5.0531,-9.4389,1.4846,-0.054978,0.29352,0.097128,2,walk,11111,walk


### Step 4. Audit check the unified frame

In [ ]:
# ============================
# STEP 4 — QA checks
# ============================
run_qa_checks(shoaib_df, SCHEMA, UNKNOWN_ID)
check_sample_integrity(shoaib_df, SCHEMA)

Subjects: 1 | Sessions: 13
Monotonic violations (groups): 0
Median Hz: 50.00 (target=50)
Rows meeting required-not-null: 100.00%
Global mapping coverage: 92.3% (unknown=9000)

Top-15 global labels:
global_activity_label
adl_desk_device       270000
posture_stationary    180000
stairs                180000
adl_food              180000
cycle                  90000
run_jog                90000
walk                   90000
other                  90000
Name: count, dtype: Int64


### Step 5. Save outputs

In [10]:
# ============================
# STEP 5 — Save
# ============================
out_path = CLEANED / "shoaib_clean_data.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
shoaib_df.to_parquet(out_path, index=False)
print("Saved →", out_path)


Saved → /home/aidan/IMU_LM_Data/data/cleaned_premerge/shoaib_clean_data.parquet
